# 01. Skill Self-Improvement (arcskill as arcagent's optional supercharger)

arcagent ships **improver-less by default**. Skills are static SKILL.md bundles until
you opt in to `arcskill` — a separate package that watches how skills perform in
production and proposes bounded, gated, signed improvements. This mirrors the
`Brain` memory seam from SPEC-041: a structural Protocol boundary, a silent no-op
default, and an optional package that plugs in without arcagent ever importing its
types (SPEC-044).

The centerpiece is not "an LLM edits a skill file." It's the *governance* around that:
an LLM judge only ranks candidate rewrites — a deterministic **golden-task eval suite**
decides acceptance via strict improvement (fix something, break nothing). Every
mutation and lifecycle transition (retire/revive) is tier-gated through an
operator-approval ladder and audited on a signed WORM chain.

**You will learn:**
- The `SkillAdapter` Protocol — the seam arcagent talks to, and its `NullSkillAdapter` no-op default
- `select_skill_adapter` — how a config setting picks Null vs `arcskill` vs a signed BYO adapter
- `ArcSkillImprover` — the concrete adapter: per-turn signals in, bounded background optimization out
- The golden-task **eval gate** — the hard, deterministic acceptance rule that overrides the LLM judge
- The Curator **lifecycle** — usage-graded retire (reversible) and operator-initiated revive
- The change-bound + code-repair path, and the tier-gated audit trail

## 1. Setup

The setup cell below is the standard Arc walkthrough preamble: it locates the repo
root, adds every package's `src/` (and `tests/` where present) to `sys.path`, and
loads any `.env` file at the root. This is identical across every notebook so you can
skim past it once you've seen it. It's what makes `arcskill` importable here even
though it isn't part of arcagent's own dependency closure — the whole point of the
seam is that it's an *optional* install.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Quick smoke test — confirm both halves of the seam import cleanly: the Protocol
lives in `arcagent`, the concrete adapter lives in `arcskill` (a separate package
arcagent never statically imports).

In [2]:
import arcagent
import arcskill

from arcagent.skilladapt.protocol import NullSkillAdapter, SkillAdapter
from arcagent.skilladapt.select import select_skill_adapter
from arcskill.improver import ArcSkillImprover, ImproverConfig

print(f"arcagent version: {arcagent.__version__}")
print(f"arcskill version: {arcskill.__version__}")
print("Loaded:", [SkillAdapter.__name__, NullSkillAdapter.__name__, ArcSkillImprover.__name__])

arcagent version: 0.15.0
arcskill version: 0.2.0
Loaded: ['SkillAdapter', 'NullSkillAdapter', 'ArcSkillImprover']


## 2. The `SkillAdapter` seam — a Protocol, not a base class

`arcagent.skilladapt.protocol.SkillAdapter` is a `@runtime_checkable` structural
Protocol. It speaks only `str` / `int` / `None` at the boundary — no arcskill type
ever crosses into arcagent. Five methods:

- `observe(*, skill_name, tool_name, status, error_type, session_id=None)` — record one tool call inside an active skill-usage span
- `on_turn_end(*, turn, outcome, session_id=None)` — close the active span at turn end
- `maybe_improve(*, insight="", session_id=None)` — trigger a bounded, eval-gated improvement pass when usage warrants it
- `review_lifecycle(*, turn)` — run the retire/revive sweep on the proactive tick
- `retired_skills() -> frozenset[str]` — names excluded from the agent's offering

`NullSkillAdapter` is the default: every method is inert. This is what `pip install
arcagent` alone runs with — the agent works end-to-end, skill improvement is a silent
no-op, and **no files are ever written**.

In [3]:
null_adapter = NullSkillAdapter()
print("Is a structural SkillAdapter:", isinstance(null_adapter, SkillAdapter))

import asyncio


async def poke_null_adapter():
    await null_adapter.observe(
        skill_name="pdf-extract", tool_name="read_pdf", status="ok", error_type=None
    )
    await null_adapter.on_turn_end(turn=1, outcome="success")
    await null_adapter.maybe_improve()
    await null_adapter.review_lifecycle(turn=1)
    return null_adapter.retired_skills()


retired = await poke_null_adapter()
print("retired_skills():", retired)
print("Every call returned None / empty — zero files written, zero side effects.")

Is a structural SkillAdapter: True
retired_skills(): frozenset()
Every call returned None / empty — zero files written, zero side effects.


## 3. Selecting an adapter — `ExtensionPoint` + `select_skill_adapter`

`arcagent.skilladapt.select` is a thin `ExtensionPoint` instance over the SPEC-047
generalized `select_extension` mechanism (the same mechanism the `Brain` memory seam
uses — one dedup'd choice-dispatch + fail-closed BYO allowlist gate, shared by every
select-one extension point in arcagent). An `ExtensionPoint` declares the four axes
that differ between seams:

In [4]:
from arcagent.extension import ExtensionPoint

import inspect
print(inspect.getsource(ExtensionPoint))

@dataclass(frozen=True)
class ExtensionPoint:
    """Declarative descriptor for one extension seam.

    Attributes:
        name: Short family name ("brain" | "skills" | ...), used in log/audit
            messages and inspection output.
        null_factory: Zero-arg callable returning the Null default (memory/improver
            off) when the setting is ``none``/``""``/``null`` or a builtin degrades.
        builtin_modules: Maps a builtin choice string to the *actual import string*
            (e.g. ``{"arcmemory": "arcmemory", "auto": "arcmemory"}`` or
            ``{"arcskill": "arcskill.improver"}``). ``auto`` degrades silently; any
            other builtin choice warns when its module is unavailable.
        builtin_builder: ``(imported_module, context) -> instance | None``. Receives
            the lazily-imported builtin module and the seam's context dict; returns
            ``None`` to degrade to the Null default. The import is lifted out into
            :func:`select_ex

And here is the actual `_SKILLADAPT_POINT` declaration arcagent ships (read directly
from source, not reproduced by hand):

In [5]:
import arcagent.skilladapt.select as skilladapt_select

print(inspect.getsource(skilladapt_select._SKILLADAPT_POINT.__class__))
point = skilladapt_select._SKILLADAPT_POINT
print("name:            ", point.name)
print("null_factory:    ", point.null_factory)
print("builtin_modules: ", dict(point.builtin_modules))
print("kind:            ", point.kind)

@dataclass(frozen=True)
class ExtensionPoint:
    """Declarative descriptor for one extension seam.

    Attributes:
        name: Short family name ("brain" | "skills" | ...), used in log/audit
            messages and inspection output.
        null_factory: Zero-arg callable returning the Null default (memory/improver
            off) when the setting is ``none``/``""``/``null`` or a builtin degrades.
        builtin_modules: Maps a builtin choice string to the *actual import string*
            (e.g. ``{"arcmemory": "arcmemory", "auto": "arcmemory"}`` or
            ``{"arcskill": "arcskill.improver"}``). ``auto`` degrades silently; any
            other builtin choice warns when its module is unavailable.
        builtin_builder: ``(imported_module, context) -> instance | None``. Receives
            the lazily-imported builtin module and the seam's context dict; returns
            ``None`` to degrade to the Null default. The import is lifted out into
            :func:`select_ex

`select_skill_adapter(setting, ...)` maps `[modules.skills] adapter` to a concrete
adapter:

- `"none"` (or unset) → `NullSkillAdapter`
- `"arcskill"` → lazy-imports `arcskill.improver` and builds an `ArcSkillImprover`; if the
  package isn't installed, it degrades to `NullSkillAdapter` with a warning rather than crashing
- a dotted class path → a user-supplied BYO adapter, refused above the `personal` tier unless
  operator-allowlisted (ASI04 — no silent trust of arbitrary code)

Let's exercise both real paths.

In [6]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    workspace = Path(tmp)

    none_adapter = select_skill_adapter("none", workspace=workspace)
    print("setting='none'     ->", type(none_adapter).__name__)
    assert isinstance(none_adapter, NullSkillAdapter)

    arcskill_adapter = select_skill_adapter(
        "arcskill",
        workspace=workspace,
        config={},
        tier="personal",
        llm=None,
        signer=None,
        approval_provider=None,
        eval_runner=None,
        audit_sink=None,
        agent_did="did:arc:demo:agent/0001",
        skill_path=None,
    )
    print("setting='arcskill' ->", type(arcskill_adapter).__name__)
    assert isinstance(arcskill_adapter, ArcSkillImprover)
    assert isinstance(arcskill_adapter, SkillAdapter)
    print("ArcSkillImprover satisfies the SkillAdapter Protocol structurally:",
          isinstance(arcskill_adapter, SkillAdapter))

setting='none'     -> NullSkillAdapter
setting='arcskill' -> ArcSkillImprover
ArcSkillImprover satisfies the SkillAdapter Protocol structurally: True


## 4. `ArcSkillImprover` — signals in, bounded background optimization out

`ArcSkillImprover(workspace, *, config=None, tier="personal", llm=None, signer=None,
eval_runner=None, mutator=None, approval_provider=None, audit_sink=None, agent_did="",
skill_path=None, reload=None, session_id="", max_concurrent=2)` is the production
`SkillAdapter`. It is *provider-free* — every external capability (LLM, signing,
sandboxed eval, operator approval, audit) arrives as an injected Protocol seam, so
`arcskill.improver` never imports `arcagent`, `arcllm`, or `arcmemory` directly
(REQ-004). arcagent's `skilladapt` wiring supplies the concrete implementations.

Internally it owns a `TraceStore` that assembles per-skill usage spans from the
primitive signals the arcagent extension forwards off the module bus:

1. `observe(skill_name=..., tool_name=..., status=..., error_type=...)` opens a span for a
   skill on first sight this turn and appends each tool call to it
2. `on_turn_end(turn=N, outcome=...)` closes every open span, persists it as JSONL, and
   bumps a per-skill usage counter
3. once a skill's usage counter crosses `config.optimize_after_uses`, `maybe_improve()`
   spawns a **bounded background task** for it — a failing optimization is caught and
   logged, never propagated to the agent loop (NFR-005)

In [7]:
import tempfile
from pathlib import Path

tmp_ws = Path(tempfile.mkdtemp())
config = ImproverConfig(optimize_after_uses=3)  # low threshold so the demo triggers fast

improver = ArcSkillImprover(
    tmp_ws,
    config=config,
    tier="personal",
    agent_did="did:arc:demo:agent/0001",
    session_id="demo-session",
)
print("tier:", improver.tier)
print("initial usage_counts:", improver._store.usage_counts)

tier: personal
initial usage_counts: {}


In [8]:
async def run_three_turns():
    for turn in range(3):
        # Each turn, the agent uses the "pdf-extract" skill for one tool call.
        await improver.observe(
            skill_name="pdf-extract",
            tool_name="read_pdf",
            status="ok",
            error_type=None,
        )
        await improver.on_turn_end(turn=turn, outcome="success")


await run_three_turns()
print("usage_counts after 3 turns:", improver._store.usage_counts)
print("turn_number:               ", improver._store.turn_number)

traces = improver._store.load_traces("pdf-extract")
print("persisted spans:            ", len(traces))
print("first span outcome:         ", traces[0].task_outcome)

usage_counts after 3 turns: {'pdf-extract': 3}
turn_number:                3
persisted spans:             3
first span outcome:          success


`optimize_after_uses=3` was crossed, so `maybe_improve()` will now spawn a background
optimization pass for `pdf-extract`. We passed no `llm=` and no `skill_path=`, so the
pass has nothing to optimize (no skill text to read, no LLM to drive) and safely
no-ops — this is the guarded path: exceptions inside `_optimize()` are caught by
`_guarded()` and logged, never raised into the caller.

In [9]:
async def trigger_and_await():
    await improver.maybe_improve()
    # give the background task a beat to run and self-discard
    await asyncio.sleep(0.05)
    await improver.aclose()  # graceful shutdown: await any still-in-flight tasks


await trigger_and_await()
print("usage_counts reset after crossing threshold:", improver._store.usage_counts)
print("No exception propagated — a failing/incomplete optimization never touches the agent loop.")

usage_counts reset after crossing threshold: {'pdf-extract': 0}
No exception propagated — a failing/incomplete optimization never touches the agent loop.


## 5. The golden-task eval gate — the HARD acceptance rule

An LLM judge (`SkillEvaluator` / `SkillReflector`, GEPA-style reflective mutation)
only **ranks** candidate rewrites. Acceptance is decided by a completely separate,
deterministic mechanism: a **golden-task pytest suite** under `<skill_dir>/evals/`,
discovered by a static AST scan (`load_suite`) — no code is imported or executed to
find the tests, only parsed.

The rule (`EvalGate._strict_improvement`) is **strict improvement**: a candidate is
accepted only if it makes **at least one previously-failing** case pass **and
regresses zero** previously-passing cases. Ties and neutral drift are rejected — this
is what stops an optimizer from "improving" a skill sideways forever.

In [10]:
from arcskill.improver.evalgate import EvalGate, GateDecision, no_suite_policy
from arcskill.improver.models import BundleView, EvalCase, EvalOutcome


class FakeEvalRunner:
    """A deterministic EvalRunner fake — the real production runner sandboxes
    execution via arcskill.hub.dry_run (Firecracker/Docker); here we just script
    canned pass/fail outcomes per BundleView.text so the gate logic is provable
    without a live sandbox."""

    def __init__(self, outcomes_by_text: dict[str, dict[str, bool]]):
        self._outcomes_by_text = outcomes_by_text

    async def run(self, view: BundleView, cases: list[EvalCase]) -> list[EvalOutcome]:
        table = self._outcomes_by_text[view.text]
        return [EvalOutcome(case_id=c.id, passed=table[c.id]) for c in cases]


cases = [EvalCase(id="t1"), EvalCase(id="t2"), EvalCase(id="t3")]


async def demo_gate(before_passes: dict[str, bool], after_passes: dict[str, bool], label: str):
    runner = FakeEvalRunner({"before": before_passes, "after": after_passes})
    gate = EvalGate(runner, min_golden_cases=1)
    decision = await gate.decide(
        before=BundleView("demo-skill", "before"),
        after=BundleView("demo-skill", "after"),
        cases=cases,
        tier="personal",
        kind="prose",
    )
    print(f"{label:28s} accepted={decision.accepted!s:5s} reason={decision.reason}")
    return decision


async def run_all_three():
    # (a) regression: t2 flips from pass -> fail. Rejected even though t3 also newly passes.
    await demo_gate(
        {"t1": True, "t2": True, "t3": False},
        {"t1": True, "t2": False, "t3": True},
        "(a) regression",
    )
    # (b) no strict improvement: nothing newly passes, nothing regresses. Rejected as a tie.
    await demo_gate(
        {"t1": True, "t2": False, "t3": False},
        {"t1": True, "t2": False, "t3": False},
        "(b) no improvement",
    )
    # (c) strict improvement: t2 flips fail -> pass, nothing regresses. Accepted.
    await demo_gate(
        {"t1": True, "t2": False, "t3": False},
        {"t1": True, "t2": True, "t3": False},
        "(c) strict improvement",
    )


await run_all_three()

(a) regression               accepted=False reason=regression on 1 golden case(s)
(b) no improvement           accepted=False reason=no strict improvement (no previously-failing case now passes)
(c) strict improvement       accepted=True  reason=strict improvement: fixed failing case(s), no regression


When a skill has **no** golden suite at all, `no_suite_policy(tier, kind)` decides
fail-closed:

- **code** mutations with no suite are blocked at **every** tier — self-modifying code
  is never accepted without a deterministic test to prove it
- **prose** mutations with no suite are allowed only at `tier="personal"` (audit-warned);
  `enterprise`/`federal` block outright

In [11]:
for tier in ("personal", "enterprise", "federal"):
    for kind in ("prose", "code"):
        decision = no_suite_policy(tier, kind)
        print(f"tier={tier:10s} kind={kind:6s} -> accepted={decision.accepted!s:5s} ({decision.reason})")

AUDIT WARN: prose mutation with no golden suite (personal tier)


tier=personal   kind=prose  -> accepted=True  (prose mutation, no suite (personal audit-warn))
tier=personal   kind=code   -> accepted=False (code mutation blocked: no golden-task suite (any tier))
tier=enterprise kind=prose  -> accepted=False (prose mutation blocked: no golden-task suite (tier=enterprise))
tier=enterprise kind=code   -> accepted=False (code mutation blocked: no golden-task suite (any tier))
tier=federal    kind=prose  -> accepted=False (prose mutation blocked: no golden-task suite (tier=federal))
tier=federal    kind=code   -> accepted=False (code mutation blocked: no golden-task suite (any tier))


## 6. Curator lifecycle — usage-graded retire (reversible) and operator revive

`SkillLifecycle` drives `active → underperforming → retired` purely from accrued
usage stats — and `retired → active` on operator-initiated revive. Retirement is a
**reversible disable**, never a destructive delete: lineage is retained so a bad call
can be undone (D-8).

`pending_retirements()` is a pure grading pass — it commits nothing. The caller
(`ArcSkillImprover.review_lifecycle`) is what gates each proposed retirement through
the tier operator-approval ladder before actually committing it via `retire()`.

In [12]:
from arcskill.improver.lifecycle import SkillLifecycle, UsageStats
from arcskill.improver.candidate_store import CandidateStore
from arcskill.improver.config import LifecycleConfig
from arcskill.improver.models import SkillTrace
from datetime import datetime, timedelta, UTC

lifecycle_ws = Path(tempfile.mkdtemp())
store = CandidateStore(lifecycle_ws)
lifecycle_config = LifecycleConfig(
    inactivity_window_days=30.0, failure_floor=0.5,
    improve_attempts_before_retire=3, min_uses_before_retire=5,
)

# A skill that hasn't been used in 60 days -> past the 30-day inactivity window.
stale_trace = SkillTrace(
    trace_id="t1", session_id="s1", skill_name="stale-skill", skill_version=0,
    turn_number=0, started_at=datetime.now(UTC) - timedelta(days=61),
    ended_at=datetime.now(UTC) - timedelta(days=60), task_outcome="success",
)
store.set_lifecycle_state("stale-skill", "active")  # register it as known


def load_traces(name: str) -> list[SkillTrace]:
    return [stale_trace] if name == "stale-skill" else []


curator = SkillLifecycle(store, lifecycle_config, load_traces=load_traces, generation_of=lambda n: 0)
pending = curator.pending_retirements()
print("pending retirements:", pending)

pending retirements: [('stale-skill', 'inactive past window')]


In [13]:
# ArcSkillImprover.review_lifecycle() would gate each of these through the tier
# approval ladder (_approval_required) before committing. Reproduce that ladder
# directly against the real ArcSkillImprover instance from section 4:
print(f"{'tier':<10} {'code':<8} {'prose':<8} {'retire':<8} {'revive':<8}")
for tier in ("personal", "enterprise", "federal"):
    probe = ArcSkillImprover(Path(tempfile.mkdtemp()), tier=tier)
    row = [probe._approval_required(kind) for kind in ("code", "prose", "retire", "revive")]
    print(f"{tier:<10} " + "".join(f"{str(r):<8}" for r in row))
print()
print("personal: never requires approval (auto + audit).")
print("enterprise: only code mutations require approval.")
print("federal: every mutation AND retire/revive requires operator approval (fail-closed if unwired).")

tier       code     prose    retire   revive  
personal   False   False   False   False   
enterprise True    False   False   False   
federal    True    True    True    True    

personal: never requires approval (auto + audit).
enterprise: only code mutations require approval.
federal: every mutation AND retire/revive requires operator approval (fail-closed if unwired).


In [14]:
event = curator.retire("stale-skill", reason=pending[0][1])
print("transition:", event.from_state, "->", event.to_state, "| reason:", event.reason)
print("state now:", curator.state("stale-skill"))

revive_event = curator.revive("stale-skill")
print("transition:", revive_event.from_state, "->", revive_event.to_state)
print("state now:", curator.state("stale-skill"), "(lineage retained, not deleted)")

transition: active -> retired | reason: inactive past window
state now: retired
transition: retired -> active
state now: active (lineage retained, not deleted)


`ArcSkillImprover.retired_skills()` is the read-side filter the agent's tool offering
consults — it's sourced from the candidate-store manifest, so it survives process
restarts.

In [15]:
offering_ws = Path(tempfile.mkdtemp())
offering_improver = ArcSkillImprover(offering_ws, tier="personal")
offering_improver._candidate_store.set_lifecycle_state("broken-skill", "retired", reason="demo")
print("retired_skills():", offering_improver.retired_skills())

retired_skills(): frozenset({'broken-skill'})


## 7. Change-bound + code-repair (context, not the centerpiece)

Not every skill fix is prose. A skill with a `scripts/` (or `src/`) directory *and* a
golden suite, whose failing traces carry a code `error_type`, is eligible for
**code repair** instead of a prose rewrite (`ArcSkillImprover._should_repair_code`):
an injected `Mutator` (production default: `LLMCodeMutator`, arcllm-backed) proposes a
patch, which must clear a **change-bound** budget *before* the costly sandboxed eval
even runs.

`ChangeBound.scheduled_edits(generation, improve_attempts_before_retire)` decays the
per-attempt edit budget on a cosine schedule across the skill's improve-attempt
budget — early attempts get room to explore, later attempts are squeezed toward
minimal, surgical fixes before the skill is retired outright. A patch that blows the
budget is rejected *and audited* before ever touching the sandbox — the expensive
step only runs on patches that are structurally plausible.

Every prior *rejected* patch is fed back to the mutator as bounded negative feedback
(`_compose_failures` / `_rejected` buffer, capped at 5 entries per skill) — this is
the SkillOpt mechanism credited with stopping the mutator from re-proposing the same
losing edit forever.

In [16]:
from arcskill.improver.guardrails import ChangeBound
from arcskill.improver.config import ChangeBoundConfig

# "personal" defaults to a flat (constant) budget; enterprise/federal use the cosine
# schedule — pick enterprise so the decay is visible.
bound = ChangeBound("enterprise", ChangeBoundConfig())
total_attempts = 4  # e.g. config.lifecycle.improve_attempts_before_retire
for attempt in range(total_attempts):
    budget = bound.scheduled_edits(attempt, total_attempts)
    print(f"attempt={attempt} -> edit budget={budget}")
print()
print("Budget shrinks (cosine decay) toward the retire-after-N-attempts ceiling —")
print("early attempts explore, later ones consolidate toward a minimal fix.")

attempt=0 -> edit budget=4
attempt=1 -> edit budget=4
attempt=2 -> edit budget=2
attempt=3 -> edit budget=2

Budget shrinks (cosine decay) toward the retire-after-N-attempts ceiling —
early attempts explore, later ones consolidate toward a minimal fix.


## 8. Audit trail — every mutation and transition is a signed WORM event

`ArcSkillImprover._emit_audit` stamps every mutation/lifecycle transition as an
`arctrust.AuditEvent` tagged with the constructed `tier` and `actor_did` (the agent's
own DID — distinct from the *operator* key that actually signs the WORM chain). We
wire in a real `arctrust` sink: `WormSink` is the current durable, Ed25519-signed,
hash-chained audit log (it consolidates what used to be two separate sink classes —
an unsigned JSONL sink and a signed hash-chain sink — into one, since every record is
now signed and chained unconditionally).

In [17]:
from arctrust import AuditEvent, WormSink, emit, generate_keypair
from arctrust.signer import InProcessSigner, ED25519

audit_ws = Path(tempfile.mkdtemp())
keypair = generate_keypair()
signer = InProcessSigner(keypair.private_key, ED25519)
chain_path = audit_ws / "skills.worm"
sink = WormSink(chain_path, signer)

audited_improver = ArcSkillImprover(
    Path(tempfile.mkdtemp()),
    tier="personal",
    agent_did="did:arc:demo:agent/0001",
    audit_sink=sink,
)
# rollback() needs a real prior candidate on disk to revert to; to keep this demo
# self-contained we call the same _emit_audit mechanism rollback() itself calls,
# proving the exact audit-event shape without staging a full candidate lineage.
audited_improver._emit_audit(
    "some-skill", "skill.mutation.rolled_back", "rolled_back",
    extra={"candidate_id": "candidate-007"},
)

lines = chain_path.read_text().strip().splitlines()
print(f"{len(lines)} record(s) written to the WORM chain")
import json
record = json.loads(lines[0])
print("seq:      ", record["seq"])
print("action:   ", record["event"]["action"])
print("actor_did:", record["event"]["actor_did"])
print("outcome:  ", record["event"]["outcome"])
print("tier:     ", record["event"]["tier"])

1 record(s) written to the WORM chain
seq:       0
action:    skill.mutation.rolled_back
actor_did: did:arc:demo:agent/0001
outcome:   rolled_back
tier:      personal


## 9. The auto-skill-creation nudge — a related, distinct mechanism

Everything above *modifies or retires an existing* skill. `arcskill.improver.nudge`
solves the opposite problem: noticing when the agent just executed a multi-tool
workflow that **no skill covers yet**, and nudging toward creating one. `NudgeEmitter`
subscribes to `agent:post_plan` (after `trace_collector` has closed the span) and
evaluates an AND-conjunction (SDD §3.7):

1. `tool_calls_ok >= 5` — the workflow is non-trivial
2. `task_outcome == "success"` — only nudge on wins
3. **novelty**: at least one of — an error was recovered from, a user correction was
   detected, or no existing skill covers this shape well (`max_existing_skill_coverage
   < 0.3`)
4. not in the session's cooldown window

Critically (ASI-09), `NudgeEmitter` **never calls `skill_manage()` or writes anything**
— it only publishes an advisory `system_message_nudge` event that `context_manager`
injects as text on the next turn. The LLM agent decides whether to act on it, and the
user confirms. Before a nudge fires, `pre_commit_dedup` runs three checks in order —
name collision → tool-sequence fingerprint match → semantic (cosine) similarity — so
the same workflow shape doesn't spam a nudge for a skill that already exists in
substance, even under a different name.

In [18]:
from arcskill.improver.nudge.nudge_emitter import NudgeEmitter
from arcskill.improver.nudge.signals import NudgeSignals

nudge_config = ImproverConfig()
emitter = NudgeEmitter(nudge_config, session_id="demo-session")


def trigger(tool_calls_ok, task_outcome, error_count, coverage):
    signals = NudgeSignals(
        tool_calls_ok=tool_calls_ok,
        task_outcome=task_outcome,
        error_count=error_count,
        user_correction_detected=False,
        max_existing_skill_coverage=coverage,
        turn_number=1,
        trace_id="t1",
    )
    return emitter._evaluate_trigger(signals)


print("5 ok calls, success, recovered an error, low coverage ->", trigger(5, "success", 1, 0.1))
print("5 ok calls, success, no errors, HIGH coverage (skill exists) ->", trigger(5, "success", 0, 0.9))
print("only 3 ok calls (below the floor of 5) ->", trigger(3, "success", 1, 0.1))
print("5 ok calls but the task itself failed ->", trigger(5, "failure", 1, 0.1))

5 ok calls, success, recovered an error, low coverage -> True
5 ok calls, success, no errors, HIGH coverage (skill exists) -> False
only 3 ok calls (below the floor of 5) -> False
5 ok calls but the task itself failed -> False


In [19]:
from arcskill.improver.nudge.dedup import compute_tool_sequence_hash, pre_commit_dedup

existing_names = {"skill-bash-grep-read"}
known_fingerprints = {compute_tool_sequence_hash(["bash", "grep", "read"])}
known_tool_lists = [["bash", "grep", "read"]]

# (a) name collision: the proposed name already exists.
dup, reason = pre_commit_dedup(
    proposed_name="skill-bash-grep-read",
    existing_names=existing_names,
    tool_sequence_hash=compute_tool_sequence_hash(["write", "edit"]),
    known_fingerprints=set(),
    candidate_tools=["write", "edit"],
    known_tool_lists=[],
)
print("(a) name collision   -> is_dup =", dup, " reason =", reason)

# (b) fingerprint match: same tool SET (order-independent), different proposed name.
dup, reason = pre_commit_dedup(
    proposed_name="skill-a-totally-different-name",
    existing_names=existing_names,
    tool_sequence_hash=compute_tool_sequence_hash(["read", "grep", "bash"]),
    known_fingerprints=known_fingerprints,
    candidate_tools=["read", "grep", "bash"],
    known_tool_lists=known_tool_lists,
)
print("(b) fingerprint match-> is_dup =", dup, " reason =", reason)

# (c) genuinely novel shape: none of the three checks fire.
dup, reason = pre_commit_dedup(
    proposed_name="skill-curl-jq",
    existing_names=existing_names,
    tool_sequence_hash=compute_tool_sequence_hash(["curl", "jq"]),
    known_fingerprints=known_fingerprints,
    candidate_tools=["curl", "jq"],
    known_tool_lists=known_tool_lists,
)
print("(c) novel, not a dup -> is_dup =", dup, " reason =", reason or "(none — nudge proceeds)")

(a) name collision   -> is_dup = True  reason = name_collision
(b) fingerprint match-> is_dup = True  reason = fingerprint_match
(c) novel, not a dup -> is_dup = False  reason = (none — nudge proceeds)


That's the whole loop: silent no-op by default, an explicit opt-in seam, per-turn
signals feeding a bounded background optimizer, a deterministic golden-task suite
that overrides the LLM's own judgment, a reversible usage-graded lifecycle, an
advisory nudge that proposes *new* skills without ever writing one itself, and a
signed audit trail on every consequential step. Nothing here trusts the model to
police itself — the gate, the tier ladder, the dedup checks, and the WORM chain do
that.